# Predicting ICU Mortality from the First 24 Hours

A small machine learning tutorial based on MIMIC-III data. My goal is to use only what we learn about a patient in their first 24 hours in the ICU to predict whether they survive the hospital stay.

Each row is one ICU stay. The target is `mortality` (1 = died in hospital, 0 = survived).
The features are age, sex, admission type, and a handful of first-24h vitals and labs.
All the heavy lifting (reducing hundreds of millions of rows down to one row per stay) was done in BigQuery ahead of this notebook — see `planning/04_bigquery_pull.sql`.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

SEED = 42          # fixed so results are repeatable
np.random.seed(SEED)

## Step 1 — Load the data

In [ ]:
# this came straight out of BigQuery (one row per ICU stay).
# works whether I saved it as parquet or csv.
# NOTE: the data file isn't in this repo (MIMIC data can't be redistributed).
# drop your own export next to this notebook, or point DATA at wherever it lives.
import os
DATA = "." # folder holding the data file
pq  = os.path.join(DATA, "icu_first24h_features.parquet")
csv = os.path.join(DATA, "icu_first24h_features.csv")
df = pd.read_parquet(pq) if os.path.exists(pq) else pd.read_csv(csv)
print(df.shape)
df.head()

In [ ]:
# quick gut check: how many died vs survived?
df["mortality"].value_counts(normalize=True)

## Step 2 — Clean it up

Quick note on the missing values, because this one actually matters and it took me a
minute to think through.

My first instinct was just to fill the blank labs with the median. That's a common practice
but here it's misleading because a lab only performed if a doctor nurse had a reason to order it. 
So the patients who have lactate value(s) skew sicker, and are a hand-picked group, so their values likely skew abnormal. 
The patients with a blank probably didn't get one because nobody was worried about their lactate values - so were likely
closer to normal, which may not map to the median value of those in this data set. 

So filling their blank with the median of the those who had the test ordered overstates
how bad off they were. The "healthy" thing to fill in would usually be lower than the
median I'm using.

I kept the median fill anyway (it's simple and reproducible), but I added a
"was this measured?" flag for every column that goes missing. That flag is the important
part: it lets the model learn that if the value wasn't actually ordered, perhaps 
it shouldn't be weighted too heavily.

In other words, the flag opens the door for the model to minimize the bias the median fill
approach may introduce. A fancier fix would be something like KNN or iterative imputation that
estimates each missing value from the other columns, but for a tree model, fill + flag
gets most of the benefit with way less complexity, and this is a simple tutorial after all :)

In [ ]:
# drop the id, it's not a feature
df = df.drop(columns=["icustay_id"])

# turn gender into 0/1 and one-hot encode the admission type
df["gender"] = (df["gender"] == "M").astype(int)
df = pd.get_dummies(df, columns=["admission_type"], drop_first=True)

# the labs and vitals are the columns that actually go missing. in a hospital,
# a missing lab usually means nobody ordered it - and *that* is a clue in itself
# (you draw lactate when you suspect shock, not routinely). so before I fill the
# blanks, I add a "was this measured?" flag for each one. 1 = it was missing.
maybe_missing = [c for c in df.columns if df[c].isna().any()]
for c in maybe_missing:
    df[c + "_missing"] = df[c].isna().astype(int)

# now fill the actual blanks with the column median (robust to outliers).
# the flags above keep the "it wasn't measured" signal even after filling.
df = df.fillna(df.median(numeric_only=True))

df.head()

## Step 3 — Split into train and test

In [ ]:
X = df.drop(columns=["mortality"])
y = df["mortality"]

# stratify keeps the same ~12% death rate in both halves
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
print(X_train.shape, X_test.shape)

## Step 4 — Train a random forest

In [ ]:
# class_weight="balanced" tells it not to ignore the rare deaths
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
model.fit(X_train, y_train)

## Step 5 — Results

In [ ]:
# predicted probability of death for each test patient
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

auc = roc_auc_score(y_test, y_prob)
print(f"AUC: {auc:.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["survived", "died"]))

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")   # the "random guessing" line
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve - ICU mortality")
plt.legend()
plt.savefig("fig_roc.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# confusion matrix - where did it get things right and wrong?
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center")
ax.set_xticks([0, 1]); ax.set_xticklabels(["survived", "died"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["survived", "died"])
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Confusion matrix")
plt.savefig("fig_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

## Results - Concerns

That confusion matrix was discouraging, despite the AUC being better than expected (0.85).

Out out of ~1,600 patients who actually died, the model only flagged 300 of them.
I envision this model being truly helpful for medical practitioners so I will explore biasing towards
false positives (i.e., those that do not die but are predicted to) to minimize false negatives (i.e., those
who do die but are predicted not to) without going too extreme and including a sizeable portion of the population.

The reason for the missed deaths is the cutoff. By default the model only calls someone "high risk" when
it's more than 50% sure they'll die. But deaths are rare (~12%), so it almost never
gets that confident, and it ends up playing it "safe" by saying "survives" most of the time.

In [ ]:
# try a bunch of cutoffs and see how many deaths we catch vs how many
# false alarms we raise. (a "false alarm" = we flagged someone who actually survived.)
from sklearn.metrics import confusion_matrix

rows = []
for t in [0.50, 0.40, 0.30, 0.25, 0.20, 0.15]:
    pred = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({
        "cutoff": t,
        "deaths_caught": tp,
        "deaths_missed": fn,
        "false_alarms": fp,
        "recall": round(tp / (tp + fn), 2),      # % of real deaths we caught
    })

pd.DataFrame(rows)

If we drop the cutoff to 0.20, we go from catching ~19% of deaths to
~61% of them. The price is more false alarms — but an ICU a false alarm likely 
means a nurse double-checks a patient, while a missed death is the worst case.
So for this kind of problem I'd happily take that trade off.

Lesson I didn't expect going in: picking the cutoff matters just as much as picking
the model. Same exact model, totally different behavior depending on where we draw the line.

In [ ]:
# let's look at the confusion matrix again at the cutoff I actually like (0.20)
y_pred_020 = (y_prob >= 0.20).astype(int)
cm2 = confusion_matrix(y_test, y_pred_020)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(cm2, cmap="Blues")
for (i, j), v in np.ndenumerate(cm2):
    ax.text(j, i, str(v), ha="center", va="center")
ax.set_xticks([0, 1]); ax.set_xticklabels(["survived", "died"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["survived", "died"])
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Confusion matrix at cutoff = 0.20")
plt.savefig("fig_confusion_020.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# which features did the forest utilize the most?
importances = (
    pd.Series(model.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(12)
)

plt.figure(figsize=(8, 6))
importances[::-1].plot.barh()
plt.title("Top features for predicting ICU mortality")
plt.xlabel("importance")
plt.savefig("fig_importance.png", dpi=150, bbox_inches="tight")
plt.show()

importances